🏗 Can One DOCX Template Be Accessed by Multiple Async Jobs?

✅ Yes, with proper handling
❌ No, if you try to modify the same Document object concurrently in memory

🧠 Why Conflicts Happen

python-docx works in-memory:

from docx import Document
doc = Document("template.docx")

If two async tasks load the same file into memory separately, they are isolated → ✅ safe.

If two tasks share the same Document object in memory → ❌ race conditions, corrupted doc, wrong output.

✅ Safe Pattern (Recommended)
1️⃣ Keep template on disk, read fresh for each job
def generate_doc(job_data):
    doc = Document("templates/ai_strategy_template.docx")  # new instance
    # replace placeholders
    doc.save(f"output_{job_data['job_id']}.docx")

Each async task gets its own Document instance

No conflict, even if 50 tasks run simultaneously

2️⃣ Optional: Copy template to temp file per job (if large templates)
import shutil
import tempfile
from docx import Document

def generate_doc(job_data):
    with tempfile.NamedTemporaryFile(suffix=".docx") as tmp:
        shutil.copy("templates/ai_strategy_template.docx", tmp.name)
        doc = Document(tmp.name)
        # replace placeholders
        doc.save(f"output_{job_data['job_id']}.docx")

Useful if you want isolated file operations

Safer if workers might accidentally share paths

Good in Celery / multi-process setups

3️⃣ Never do this:
# ❌ SHARING THE SAME OBJECT ACROSS TASKS
shared_doc = Document("template.docx")

def generate_doc(job_data):
    # concurrent access → may corrupt doc
    shared_doc.paragraphs[0].text = job_data["title"]
    shared_doc.save(...)

This will break in async / multi-threaded environments

🔑 Key Rule

Always load a new Document instance from the template for each job.
Template files on disk are read-only for workers.
Each worker handles its own in-memory copy.

🏗 Scaling Implications

1 template file → used safely by 100+ concurrent jobs

Memory usage grows with the number of active jobs

For very large templates → use temp files per job to avoid disk I/O bottlenecks

⚡ Bonus Tips for High-Scale Systems

Store templates on shared storage (S3, NFS) → workers load independently

Preload small templates in memory per worker if reused frequently

Never modify template in place

Return unique output filenames per job (report_{job_id}.docx)

In short:

✅ Multiple async jobs can safely use the same template
✅ Each job must load a fresh copy
❌ Do not share a single Document object in memory across tasks